In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from scipy.signal import hilbert

# ============================================================
# MULTI-SUBJECT HUP STRICT HFO DETECTION
# Runs for:
#   sub-HUP126
#   sub-HUP164
#   sub-HUP130
#   sub-HUP157
#
# Outputs final merged LLM-ready JSON into:
#   D:\LLM_outputs_for_subjects\LLM_unified_HUP_<SUBJECTNAME>\hfo\
# ============================================================

In [2]:
SUBJECT_CONFIGS = [
    {
        "subject": "sub-HUP126",
        "preproc_root": Path(r"D:\HUP126_hfo_preproc"),
        "detect_root": Path(r"D:\HUP126_hfo_detect_strict"),
        "session": "presurgery",
        "task": "interictal",
        "acq": "ecog",
        "runs": ["01", "02"],   # adjust if needed
    },
    {
        "subject": "sub-HUP164",
        "preproc_root": Path(r"D:\HUP164_hfo_preproc"),
        "detect_root": Path(r"D:\HUP164_hfo_detect_strict"),
        "session": "presurgery",
        "task": "interictal",
        "acq": "seeg",
        "runs": ["01", "02"],   # adjust if needed
    },
    {
        "subject": "sub-HUP130",
        "preproc_root": Path(r"D:\HUP130_hfo_preproc"),
        "detect_root": Path(r"D:\HUP130_hfo_detect_strict"),
        "session": "presurgery",
        "task": "interictal",
        "acq": "seeg",
        "runs": ["01", "02"],   # adjust if needed
    },
    {
        "subject": "sub-HUP157",
        "preproc_root": Path(r"D:\HUP157_hfo_preproc"),
        "detect_root": Path(r"D:\HUP157_hfo_detect_strict"),
        "session": "presurgery",
        "task": "interictal",
        "acq": "seeg",
        "runs": ["01", "02"],   # adjust if needed
    },
]

LLM_UNIFIED_ROOT_BASE = Path(r"D:\LLM_outputs_for_subjects")
LLM_UNIFIED_ROOT_BASE.mkdir(parents=True, exist_ok=True)

# ============================================================
# STRICT DETECTION SETTINGS
# ============================================================
ENV_SMOOTH_MS = 3.0
Z_THRESH_RIPPLE = 5.5
Z_THRESH_FR = 6.0
MIN_DUR_MS_RIPPLE = 8.0
MIN_DUR_MS_FR = 8.0
MAX_DUR_MS_RIPPLE = 120.0
MAX_DUR_MS_FR = 80.0
MERGE_GAP_MS = 6.0

MIN_CYCLES_RIPPLE = 4
MIN_CYCLES_FR = 4

FRANDR_MS = 25.0

MAX_FR_EVENTS_PER_CHANNEL_EPOCH = 12
MAX_RIPPLE_EVENTS_PER_CHANNEL_EPOCH = 20

HFO_AREA_PERCENTILE = 95.0



In [3]:
def bids_run_prefix(subject: str, session: str, task: str, acq: str, run: str) -> str:
    return f"{subject}_ses-{session}_task-{task}_acq-{acq}_run-{run}"


def robust_z(x: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return 0.6745 * (x - med) / (mad + eps)


def moving_average(x: np.ndarray, win: int) -> np.ndarray:
    if win <= 1:
        return x
    k = np.ones(win, dtype=np.float64) / win
    return np.convolve(x, k, mode="same")


def segments_from_mask(mask: np.ndarray):
    idx = np.flatnonzero(mask.astype(np.int8))
    if idx.size == 0:
        return []
    cuts = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[cuts + 1]]
    ends = np.r_[idx[cuts] + 1, idx[-1] + 1]
    return [(int(s), int(e)) for s, e in zip(starts, ends)]


def merge_close_segments(segs, gap: int):
    if not segs:
        return []
    out = [list(segs[0])]
    for s, e in segs[1:]:
        if s - out[-1][1] <= gap:
            out[-1][1] = e
        else:
            out.append([s, e])
    return [(int(a), int(b)) for a, b in out]


def count_cycles(x_seg: np.ndarray) -> int:
    x_seg = x_seg - np.mean(x_seg)
    s = np.sign(x_seg)
    s[s == 0] = 1
    zero_crossings = np.sum(np.diff(s) != 0)
    return int(zero_crossings // 2)


def detect_events_from_epochs(
    epo: mne.Epochs,
    band_name: str,
    z_thresh: float,
    min_dur_ms: float,
    max_dur_ms: float,
    min_cycles: int,
    max_events_per_channel_epoch: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data = epo.get_data()
    fs = float(epo.info["sfreq"])
    ch_names = list(epo.ch_names)
    env_win = max(1, int(ENV_SMOOTH_MS * fs / 1000.0))
    merge_gap = max(0, int(MERGE_GAP_MS * fs / 1000.0))
    min_len = max(1, int(min_dur_ms * fs / 1000.0))
    max_len = max(1, int(max_dur_ms * fs / 1000.0))
    epoch_starts = epo.events[:, 0]

    rows, qc_rows, reject_rows = [], [], []
    for ei in range(data.shape[0]):
        ev_count_epoch = 0
        ep_start = int(epoch_starts[ei])

        for ci in range(data.shape[1]):
            x = data[ei, ci].astype(np.float64, copy=False)
            env = np.abs(hilbert(x))
            env = moving_average(env, env_win)
            z = robust_z(env)
            mask = z >= z_thresh
            segs = merge_close_segments(segments_from_mask(mask), merge_gap)

            cand_events = []
            for s, e in segs:
                L = e - s
                if L < min_len or L > max_len:
                    continue

                x_seg = x[s:e]
                n_cycles = count_cycles(x_seg)
                if n_cycles < min_cycles:
                    continue

                pk = s + int(np.argmax(z[s:e]))
                cand_events.append(
                    {
                        "epoch": int(ei),
                        "epoch_start": ep_start,
                        "band": band_name,
                        "channel": ch_names[ci],
                        "start_samp": int(s),
                        "end_samp": int(e),
                        "peak_samp": int(pk),
                        "peak_z": float(z[pk]),
                        "duration_ms": float((L / fs) * 1000.0),
                        "n_cycles": int(n_cycles),
                        "start_sec": float(s / fs),
                        "end_sec": float(e / fs),
                        "peak_sec": float(pk / fs),
                    }
                )

            if len(cand_events) > max_events_per_channel_epoch:
                reject_rows.append(
                    {
                        "epoch": int(ei),
                        "epoch_start": ep_start,
                        "band": band_name,
                        "channel": ch_names[ci],
                        "n_rejected_events": len(cand_events),
                        "reason": f"too_many_events_in_epoch_channel>{max_events_per_channel_epoch}",
                    }
                )
                cand_events = []

            ev_count_epoch += len(cand_events)
            rows.extend(cand_events)

        qc_rows.append({"epoch": int(ei), "epoch_start": ep_start, f"n_{band_name}": int(ev_count_epoch)})

    return pd.DataFrame(rows), pd.DataFrame(qc_rows), pd.DataFrame(reject_rows)


def label_frandr_strict(ripple_events: pd.DataFrame, fr_events: pd.DataFrame, fs: float, frandr_ms: float) -> pd.DataFrame:
    cols = [
        "epoch_start",
        "channel",
        "ripple_event_id",
        "fr_event_id",
        "delta_ms",
        "overlap_ms",
    ]
    if ripple_events.empty or fr_events.empty:
        return pd.DataFrame(columns=cols)

    tol = int(frandr_ms * fs / 1000.0)
    rip = ripple_events.reset_index(drop=False).rename(columns={"index": "ripple_event_id"})
    fr = fr_events.reset_index(drop=False).rename(columns={"index": "fr_event_id"})
    out_rows = []

    for (epoch_start, ch), rip_g in rip.groupby(["epoch_start", "channel"]):
        fr_g = fr[(fr["epoch_start"] == epoch_start) & (fr["channel"] == ch)]
        if fr_g.empty:
            continue

        for _, r in rip_g.iterrows():
            rs, re, rpk = int(r["start_samp"]), int(r["end_samp"]), int(r["peak_samp"])
            candidates = []
            for _, f in fr_g.iterrows():
                fs_, fe_, fpk = int(f["start_samp"]), int(f["end_samp"]), int(f["peak_samp"])
                delta = abs(fpk - rpk)
                overlap = max(0, min(re, fe_) - max(rs, fs_))
                if delta <= tol and overlap > 0:
                    candidates.append((delta, overlap, int(f["fr_event_id"]), fpk))

            if not candidates:
                continue

            candidates.sort(key=lambda x: (x[0], -x[1]))
            delta, overlap, fr_id, fpk = candidates[0]
            out_rows.append(
                {
                    "epoch_start": int(epoch_start),
                    "channel": str(ch),
                    "ripple_event_id": int(r["ripple_event_id"]),
                    "fr_event_id": int(fr_id),
                    "delta_ms": float((int(fpk) - rpk) / fs * 1000.0),
                    "overlap_ms": float(overlap / fs * 1000.0),
                }
            )

    return pd.DataFrame(out_rows, columns=cols)


def compute_rates_per_min(events_df: pd.DataFrame, epoch_len_sec: float, n_epochs: int, band_label: str) -> pd.DataFrame:
    total_min = (epoch_len_sec * n_epochs) / 60.0
    if events_df.empty:
        return pd.DataFrame(columns=["channel", "band", "count", "rate_per_min"])
    grp = events_df.groupby("channel", as_index=False).size().rename(columns={"size": "count"})
    grp["band"] = band_label
    grp["rate_per_min"] = grp["count"] / total_min
    return grp[["channel", "band", "count", "rate_per_min"]]


def hfo_area_by_percentile(rates_df: pd.DataFrame, band: str, percentile: float = 95.0) -> pd.DataFrame:
    df = rates_df[rates_df["band"] == band].copy()
    if df.empty:
        df["threshold"] = np.nan
        df["is_hfo_area"] = False
        return df
    vals = df["rate_per_min"].to_numpy()
    vals = vals[vals > 0]
    if vals.size == 0:
        df["threshold"] = 0.0
        df["is_hfo_area"] = False
        return df
    thr = float(np.percentile(vals, percentile))
    df["threshold"] = thr
    df["is_hfo_area"] = df["rate_per_min"] >= thr
    return df


In [4]:
def process_one_run(cfg: dict, run: str) -> None:
    subject = cfg["subject"]
    session = cfg["session"]
    task = cfg["task"]
    acq = cfg["acq"]
    preproc_root = cfg["preproc_root"]
    detect_root = cfg["detect_root"]

    prefix = bids_run_prefix(subject, session, task, acq, run)
    in_dir = preproc_root / subject
    out_dir = detect_root / subject
    out_dir.mkdir(parents=True, exist_ok=True)

    ripple_path = in_dir / f"{prefix}_ripple_epo.fif"
    fast_path = in_dir / f"{prefix}_fast_epo.fif"

    if not ripple_path.exists():
        print(f"[WARN] Missing ripple epochs: {ripple_path}")
        return
    if not fast_path.exists():
        print(f"[WARN] Missing fast epochs: {fast_path}")
        return

    ripple_epo = mne.read_epochs(ripple_path, preload=True, verbose="ERROR")
    fast_epo = mne.read_epochs(fast_path, preload=True, verbose="ERROR")

    fs = float(ripple_epo.info["sfreq"])
    if float(fast_epo.info["sfreq"]) != fs:
        raise RuntimeError(f"Ripple and fast epoch sampling rates do not match for {prefix}")

    rip_starts = ripple_epo.events[:, 0]
    fast_starts = fast_epo.events[:, 0]
    common = np.intersect1d(rip_starts, fast_starts)
    ripple_epo = ripple_epo[np.isin(rip_starts, common)]
    fast_epo = fast_epo[np.isin(fast_starts, common)]

    n_epochs = len(ripple_epo)
    if n_epochs == 0:
        print(f"[WARN] No common epochs found for {prefix}")
        return

    epoch_len_sec = float(ripple_epo.times[-1] - ripple_epo.times[0])

    ripple_events, ripple_qc, ripple_rejects = detect_events_from_epochs(
        ripple_epo,
        band_name="ripple",
        z_thresh=Z_THRESH_RIPPLE,
        min_dur_ms=MIN_DUR_MS_RIPPLE,
        max_dur_ms=MAX_DUR_MS_RIPPLE,
        min_cycles=MIN_CYCLES_RIPPLE,
        max_events_per_channel_epoch=MAX_RIPPLE_EVENTS_PER_CHANNEL_EPOCH,
    )

    fr_events, fr_qc, fr_rejects = detect_events_from_epochs(
        fast_epo,
        band_name="fr",
        z_thresh=Z_THRESH_FR,
        min_dur_ms=MIN_DUR_MS_FR,
        max_dur_ms=MAX_DUR_MS_FR,
        min_cycles=MIN_CYCLES_FR,
        max_events_per_channel_epoch=MAX_FR_EVENTS_PER_CHANNEL_EPOCH,
    )

    frandr_pairs = label_frandr_strict(ripple_events, fr_events, fs, FRANDR_MS)

    rates_ripple = compute_rates_per_min(ripple_events, epoch_len_sec, n_epochs, "ripple")
    rates_fr = compute_rates_per_min(fr_events, epoch_len_sec, n_epochs, "fr")

    if frandr_pairs.empty:
        rates_frandr = pd.DataFrame(columns=["channel", "band", "count", "rate_per_min"])
    else:
        grp = frandr_pairs.groupby("channel", as_index=False).size().rename(columns={"size": "count"})
        grp["band"] = "frandr"
        grp["rate_per_min"] = grp["count"] / ((epoch_len_sec * n_epochs) / 60.0)
        rates_frandr = grp[["channel", "band", "count", "rate_per_min"]]

    rates = pd.concat([rates_ripple, rates_fr, rates_frandr], ignore_index=True)
    fr_area = hfo_area_by_percentile(rates, "fr", HFO_AREA_PERCENTILE)
    frandr_area = hfo_area_by_percentile(rates, "frandr", HFO_AREA_PERCENTILE)

    ripple_events.to_csv(out_dir / f"{prefix}_ripple_events_strict.csv", index=False)
    fr_events.to_csv(out_dir / f"{prefix}_fr_events_strict.csv", index=False)
    frandr_pairs.to_csv(out_dir / f"{prefix}_frandr_pairs_strict.csv", index=False)
    rates.to_csv(out_dir / f"{prefix}_hfo_rates_strict.csv", index=False)
    fr_area.to_csv(out_dir / f"{prefix}_hfo_area_fr_95p_strict.csv", index=False)
    frandr_area.to_csv(out_dir / f"{prefix}_hfo_area_frandr_95p_strict.csv", index=False)

    qc = pd.DataFrame({"epoch": np.arange(n_epochs), "epoch_start": ripple_epo.events[:, 0].astype(int)})
    if not ripple_qc.empty:
        qc = qc.merge(ripple_qc, on=["epoch", "epoch_start"], how="left")
    if not fr_qc.empty:
        qc = qc.merge(fr_qc, on=["epoch", "epoch_start"], how="left")
    qc = qc.fillna(0)
    qc.to_csv(out_dir / f"{prefix}_hfo_qc_strict.csv", index=False)

    rejects = pd.concat([ripple_rejects, fr_rejects], ignore_index=True)
    rejects.to_csv(out_dir / f"{prefix}_hfo_rejects_strict.csv", index=False)

    summary = {
        "subject": subject,
        "run": run,
        "sampling_frequency_hz": fs,
        "detector_mode": "strict",
        "strict_rules": {
            "z_thresh_ripple": Z_THRESH_RIPPLE,
            "z_thresh_fr": Z_THRESH_FR,
            "min_dur_ms_ripple": MIN_DUR_MS_RIPPLE,
            "min_dur_ms_fr": MIN_DUR_MS_FR,
            "max_dur_ms_ripple": MAX_DUR_MS_RIPPLE,
            "max_dur_ms_fr": MAX_DUR_MS_FR,
            "min_cycles_ripple": MIN_CYCLES_RIPPLE,
            "min_cycles_fr": MIN_CYCLES_FR,
            "frandr_ms": FRANDR_MS,
            "max_fr_events_per_channel_epoch": MAX_FR_EVENTS_PER_CHANNEL_EPOCH,
            "max_ripple_events_per_channel_epoch": MAX_RIPPLE_EVENTS_PER_CHANNEL_EPOCH,
        },
        "n_epochs_used": n_epochs,
        "ripple_events": int(len(ripple_events)),
        "fr_events": int(len(fr_events)),
        "frandr_pairs": int(len(frandr_pairs)),
        "n_rejected_burst_rows": int(len(rejects)),
        "top_fr_channels": rates_fr.sort_values("rate_per_min", ascending=False).head(10).to_dict(orient="records"),
        "top_frandr_channels": rates_frandr.sort_values("rate_per_min", ascending=False).head(10).to_dict(orient="records"),
        "fr_area_channels_95p": fr_area.loc[fr_area["is_hfo_area"], "channel"].astype(str).tolist(),
        "frandr_area_channels_95p": frandr_area.loc[frandr_area["is_hfo_area"], "channel"].astype(str).tolist(),
    }

    with open(out_dir / f"{prefix}_hfo_summary_strict.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"\n=== {prefix} [STRICT] ===")
    print(f"epochs used          : {n_epochs}")
    print(f"ripple events        : {len(ripple_events)}")
    print(f"FR events            : {len(fr_events)}")
    print(f"FRandR pairs         : {len(frandr_pairs)}")
    print(f"burst rejections     : {len(rejects)}")

In [5]:
def safe_load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def collect_run_summary_jsons(subject_dir: Path):
    return sorted(subject_dir.glob("*_hfo_summary_strict.json"))


def aggregate_channel_stats(run_summaries):
    channel_stats = defaultdict(
        lambda: {
            "fr_rates_per_run": {},
            "frandr_rates_per_run": {},
            "fr_counts_per_run": {},
            "frandr_counts_per_run": {},
            "appears_in_top_fr_runs": 0,
            "appears_in_top_frandr_runs": 0,
            "appears_in_fr_area_95p_runs": 0,
            "appears_in_frandr_area_95p_runs": 0,
        }
    )

    for run_summary in run_summaries:
        run_id = run_summary["run"]

        for item in run_summary.get("top_fr_channels", []):
            ch = item["channel"]
            channel_stats[ch]["fr_rates_per_run"][run_id] = item.get("rate_per_min", 0.0)
            channel_stats[ch]["fr_counts_per_run"][run_id] = item.get("count", 0)
            channel_stats[ch]["appears_in_top_fr_runs"] += 1

        for item in run_summary.get("top_frandr_channels", []):
            ch = item["channel"]
            channel_stats[ch]["frandr_rates_per_run"][run_id] = item.get("rate_per_min", 0.0)
            channel_stats[ch]["frandr_counts_per_run"][run_id] = item.get("count", 0)
            channel_stats[ch]["appears_in_top_frandr_runs"] += 1

        for ch in run_summary.get("fr_area_channels_95p", []):
            channel_stats[ch]["appears_in_fr_area_95p_runs"] += 1

        for ch in run_summary.get("frandr_area_channels_95p", []):
            channel_stats[ch]["appears_in_frandr_area_95p_runs"] += 1

    aggregated = []
    for ch, stats in channel_stats.items():
        fr_rate_values = list(stats["fr_rates_per_run"].values())
        frandr_rate_values = list(stats["frandr_rates_per_run"].values())

        aggregated.append(
            {
                "channel": ch,
                "fr_rate_mean": sum(fr_rate_values) / len(fr_rate_values) if fr_rate_values else 0.0,
                "fr_rate_max": max(fr_rate_values) if fr_rate_values else 0.0,
                "frandr_rate_mean": sum(frandr_rate_values) / len(frandr_rate_values) if frandr_rate_values else 0.0,
                "frandr_rate_max": max(frandr_rate_values) if frandr_rate_values else 0.0,
                "fr_runs_present": sorted(stats["fr_rates_per_run"].keys()),
                "frandr_runs_present": sorted(stats["frandr_rates_per_run"].keys()),
                "appears_in_top_fr_runs": stats["appears_in_top_fr_runs"],
                "appears_in_top_frandr_runs": stats["appears_in_top_frandr_runs"],
                "appears_in_fr_area_95p_runs": stats["appears_in_fr_area_95p_runs"],
                "appears_in_frandr_area_95p_runs": stats["appears_in_frandr_area_95p_runs"],
                "fr_rates_per_run": stats["fr_rates_per_run"],
                "frandr_rates_per_run": stats["frandr_rates_per_run"],
                "fr_counts_per_run": stats["fr_counts_per_run"],
                "frandr_counts_per_run": stats["frandr_counts_per_run"],
            }
        )

    aggregated.sort(
        key=lambda x: (
            x["appears_in_frandr_area_95p_runs"],
            x["appears_in_top_frandr_runs"],
            x["frandr_rate_mean"],
            x["appears_in_fr_area_95p_runs"],
            x["appears_in_top_fr_runs"],
            x["fr_rate_mean"],
        ),
        reverse=True,
    )

    return aggregated


def build_consensus_summary(aggregated_channels, n_runs):
    strong_consensus = []
    moderate_consensus = []
    single_run_only = []

    for ch in aggregated_channels:
        frandr_area_hits = ch["appears_in_frandr_area_95p_runs"]
        fr_area_hits = ch["appears_in_fr_area_95p_runs"]
        frandr_top_hits = ch["appears_in_top_frandr_runs"]
        fr_top_hits = ch["appears_in_top_fr_runs"]

        if frandr_area_hits >= 1 and (frandr_top_hits >= 2 or fr_area_hits >= 2 or fr_top_hits >= 2):
            strong_consensus.append(ch["channel"])
        elif frandr_top_hits >= 1 or fr_area_hits >= 1 or fr_top_hits >= 2:
            moderate_consensus.append(ch["channel"])
        else:
            single_run_only.append(ch["channel"])

    return {
        "n_runs_aggregated": n_runs,
        "strong_consensus_channels": strong_consensus,
        "moderate_consensus_channels": moderate_consensus,
        "single_run_only_channels": single_run_only,
    }


def build_subject_json(subject_id: str, run_summaries: list):
    if not run_summaries:
        raise ValueError(f"No run summary JSON files found for {subject_id}.")

    detector_modes = sorted(set(r.get("detector_mode", "unknown") for r in run_summaries))
    sampling_freqs = sorted(set(r.get("sampling_frequency_hz", None) for r in run_summaries))

    total_epochs = sum(r.get("n_epochs_used", 0) for r in run_summaries)
    total_ripple = sum(r.get("ripple_events", 0) for r in run_summaries)
    total_fr = sum(r.get("fr_events", 0) for r in run_summaries)
    total_frandr = sum(r.get("frandr_pairs", 0) for r in run_summaries)
    total_rejected = sum(r.get("n_rejected_burst_rows", 0) for r in run_summaries)

    aggregated_channels = aggregate_channel_stats(run_summaries)
    consensus = build_consensus_summary(aggregated_channels, len(run_summaries))

    return {
        "subject_id": subject_id,
        "module": "hfo_detection",
        "dataset": "HUP",
        "input_type": "preprocessed_or_ready_detector_outputs",
        "detector_modes_present": detector_modes,
        "sampling_frequency_hz_values": sampling_freqs,
        "n_runs": len(run_summaries),
        "run_ids": [r.get("run", "unknown") for r in run_summaries],
        "overall_summary": {
            "total_epochs_used": total_epochs,
            "total_ripple_events": total_ripple,
            "total_fr_events": total_fr,
            "total_frandr_pairs": total_frandr,
            "total_rejected_burst_rows": total_rejected,
        },
        "strict_rules": run_summaries[0].get("strict_rules", {}),
        "consensus_summary": consensus,
        "aggregated_channel_evidence": aggregated_channels,
        "run_summaries": run_summaries,
        "llm_notes": [
            "Use only channels present in this JSON.",
            "Do not infer seizure concordance from HFO data alone.",
            "Prioritize FRandR and repeated cross-run evidence when discussing salient channels.",
            "State uncertainty when dominant channels differ across runs.",
        ],
    }


def export_subject_hfo_json(cfg: dict):
    subject_id = cfg["subject"]
    detect_subject_dir = cfg["detect_root"] / subject_id

    llm_subject_root = LLM_UNIFIED_ROOT_BASE / f"LLM_unified_HUP_{subject_id.replace('sub-', '')}"
    llm_hfo_dir = llm_subject_root / "hfo"
    llm_hfo_dir.mkdir(parents=True, exist_ok=True)

    output_json = llm_hfo_dir / f"{subject_id}_hfo_subject_summary_for_llm.json"

    summary_paths = collect_run_summary_jsons(detect_subject_dir)
    print(f"{subject_id}: found {len(summary_paths)} run summary JSON file(s).")

    if not summary_paths:
        print(f"[WARN] No HFO summary JSONs found for {subject_id}")
        return

    run_summaries = [safe_load_json(p) for p in summary_paths]
    subject_json = build_subject_json(subject_id, run_summaries)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(subject_json, f, indent=2)

    print(f"Saved merged LLM-ready HFO JSON to:\n{output_json}")
    print("Strong consensus channels:", subject_json["consensus_summary"]["strong_consensus_channels"])
    print("Moderate consensus channels:", subject_json["consensus_summary"]["moderate_consensus_channels"][:10])

In [6]:
def main():
    for cfg in SUBJECT_CONFIGS:
        subject = cfg["subject"]
        print("\n" + "=" * 70)
        print(f"PROCESSING SUBJECT: {subject}")
        print("=" * 70)

        for run in cfg["runs"]:
            process_one_run(cfg, run)

        export_subject_hfo_json(cfg)

    print("\nDone.")


if __name__ == "__main__":
    main()


PROCESSING SUBJECT: sub-HUP126

=== sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-01 [STRICT] ===
epochs used          : 26
ripple events        : 106
FR events            : 36
FRandR pairs         : 25
burst rejections     : 0

=== sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-02 [STRICT] ===
epochs used          : 32
ripple events        : 437
FR events            : 143
FRandR pairs         : 104
burst rejections     : 0
sub-HUP126: found 2 run summary JSON file(s).
Saved merged LLM-ready HFO JSON to:
D:\LLM_outputs_for_subjects\LLM_unified_HUP_HUP126\hfo\sub-HUP126_hfo_subject_summary_for_llm.json
Strong consensus channels: ['LDAH4']
Moderate consensus channels: ['LDMH2', 'LDMH1', 'LDMH3', 'LDMH4', 'LDAH2', 'LDA1', 'LDAH3', 'LDAH1', 'LDA2', 'LDAH5']

PROCESSING SUBJECT: sub-HUP164

=== sub-HUP164_ses-presurgery_task-interictal_acq-seeg_run-01 [STRICT] ===
epochs used          : 43
ripple events        : 165
FR events            : 77
FRandR pairs         : 36
burst